# Step 1: Libraries & Prerequisites Setup
We will start by importing key PyTorch components, utilities, and Hugging Face's `transformers` module to leverage pretrained Vision and Language models.

In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from typing import Optional, List, Tuple

# We will use Hugging Face for pretrained Vision (e.g., CLIP) and Language models (e.g., LLaMA / OPT)
import transformers
from transformers import AutoModel, AutoTokenizer

print("PyTorch version:", torch.__version__)
print("Transformers version:", transformers.__version__)

PyTorch version: 2.11.0+cu130
Transformers version: 5.22.0


# Step 2: The Perceiver Resampler
The Perceiver Resampler receives visual feature maps from the vision encoder and outputs a fixed number of visual tokens. It uses learned latent queries to query the visual features via Cross-Attention, optionally incorporating temporal embeddings for video inputs.

In [3]:
class PerceiverAttention(nn.Module):
    def __init__(self, dim: int, dim_head: int = 64, heads: int = 8):
        super().__init__()
        self.scale = dim_head ** -0.5
        self.heads = heads
        self.dim_head = dim_head
        inner_dim = dim_head * heads

        self.norm_media = nn.LayerNorm(dim)
        self.norm_latents = nn.LayerNorm(dim)

        self.to_q = nn.Linear(dim, inner_dim, bias=False)
        self.to_kv = nn.Linear(dim, inner_dim * 2, bias=False)
        self.to_out = nn.Linear(inner_dim, dim, bias=False)

    def forward(self, x: torch.Tensor, latents: torch.Tensor) -> torch.Tensor:
        """
        x: visual features [batch, seq_len, dim]
        latents: learned latent queries [batch, num_queries, dim]
        """
        x = self.norm_media(x)
        latents = self.norm_latents(latents)

        b, n, _, h = *latents.shape, self.heads

        q = self.to_q(latents)
        # Key and Value are derived from concatenating latents and visual features
        kv_input = torch.cat((x, latents), dim=1)
        k, v = self.to_kv(kv_input).chunk(2, dim=-1)

        # Reshape for multi-head attention: specify self.dim_head explicitly to avoid multiple -1s
        q = q.view(b, n, h, self.dim_head).transpose(1, 2)
        k = k.view(b, -1, h, self.dim_head).transpose(1, 2)
        v = v.view(b, -1, h, self.dim_head).transpose(1, 2)

        # Attention calculation
        dots = torch.matmul(q, k.transpose(-1, -2)) * self.scale
        attn = dots.softmax(dim=-1)

        out = torch.matmul(attn, v)
        out = out.transpose(1, 2).contiguous().view(b, n, -1)
        return self.to_out(out)

class PerceiverResampler(nn.Module):
    def __init__(self, dim: int, depth: int = 6, dim_head: int = 64, heads: int = 8, num_queries: int = 64):
        super().__init__()
        # Learned latent queries
        self.latents = nn.Parameter(torch.randn(num_queries, dim))

        self.layers = nn.ModuleList([])
        for _ in range(depth):
            self.layers.append(nn.ModuleList([
                PerceiverAttention(dim=dim, dim_head=dim_head, heads=heads),
                nn.Sequential(
                    nn.LayerNorm(dim),
                    nn.Linear(dim, dim * 4),
                    nn.GELU(),
                    nn.Linear(dim * 4, dim)
                )
            ]))
        self.norm = nn.LayerNorm(dim)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        b = x.shape[0]
        # Tile the latents across batch dimension
        latents = self.latents.unsqueeze(0).repeat(b, 1, 1)

        for attn, ff in self.layers:
            latents = attn(x, latents) + latents
            latents = ff(latents) + latents

        return self.norm(latents)

# Quick test
test_resampler = PerceiverResampler(dim=512, depth=2, num_queries=64)
test_input = torch.randn(2, 256, 512) # batch=2, 256 visual tokens, feature_dim=512
output = test_resampler(test_input)
print("Input shape:", test_input.shape)
print("Output shape:", output.shape) # Expected: [2, 64, 512]

Input shape: torch.Size([2, 256, 512])
Output shape: torch.Size([2, 64, 512])


# Step 3: Gated Cross-Attention Dense Layers
These layers are interleaved into the pre-trained Language Model to merge visual information into the text-generation process. We use a tanh-gated mechanism initialized to 0, which ensures that at initialization, the language model outputs are unaffected by the visual components.

In [4]:
class GatedCrossAttention(nn.Module):
    def __init__(self, dim: int, dim_visual: int, heads: int = 8, dim_head: int = 64):
        super().__init__()
        self.heads = heads
        self.scale = dim_head ** -0.5
        self.dim_head = dim_head
        inner_dim = dim_head * heads

        self.norm_text = nn.LayerNorm(dim)
        self.norm_visual = nn.LayerNorm(dim_visual)

        self.to_q = nn.Linear(dim, inner_dim, bias=False)
        self.to_kv = nn.Linear(dim_visual, inner_dim * 2, bias=False)
        self.to_out = nn.Linear(inner_dim, dim, bias=False)

        # Tanh gate initialized to 0
        self.tanh_gate = nn.Parameter(torch.zeros(1))

    def forward(self, x: torch.Tensor, y: torch.Tensor) -> torch.Tensor:
        """
        x: text features [batch, text_len, dim]
        y: visual features from resampler [batch, visual_len, dim_visual]
        """
        residual = x
        x = self.norm_text(x)
        y = self.norm_visual(y)

        b, n, h = x.shape[0], x.shape[1], self.heads

        q = self.to_q(x).view(b, n, h, self.dim_head).transpose(1, 2)
        k, v = self.to_kv(y).chunk(2, dim=-1)
        k = k.view(b, -1, h, self.dim_head).transpose(1, 2)
        v = v.view(b, -1, h, self.dim_head).transpose(1, 2)

        # Cross-Attention
        dots = torch.matmul(q, k.transpose(-1, -2)) * self.scale
        attn = dots.softmax(dim=-1)

        out = torch.matmul(attn, v)
        out = out.transpose(1, 2).contiguous().view(b, n, -1)
        out = self.to_out(out)

        # Apply tanh gating on the update
        return residual + out * self.tanh_gate.tanh()

class GatedFeedForward(nn.Module):
    def __init__(self, dim: int, ff_mult: int = 4):
        super().__init__()
        self.norm = nn.LayerNorm(dim)
        self.net = nn.Sequential(
            nn.Linear(dim, dim * ff_mult),
            nn.GELU(),
            nn.Linear(dim * ff_mult, dim)
        )
        self.tanh_gate = nn.Parameter(torch.zeros(1))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        residual = x
        x = self.norm(x)
        out = self.net(x)
        return residual + out * self.tanh_gate.tanh()

# Quick test to verify gating works at initialization
test_cross_attn = GatedCrossAttention(dim=768, dim_visual=512)
test_ff = GatedFeedForward(dim=768)

text_in = torch.randn(2, 20, 768)
vis_in = torch.randn(2, 64, 512)

attn_out = test_cross_attn(text_in, vis_in)
ff_out = test_ff(attn_out)

# Because tanh_gate is 0, outputs should be identical to text_in
print("Input equals Gated Attention output at initialization:", torch.allclose(text_in, attn_out, atol=1e-6))
print("Input equals Gated Feed-Forward output at initialization:", torch.allclose(text_in, ff_out, atol=1e-6))

Input equals Gated Attention output at initialization: True
Input equals Gated Feed-Forward output at initialization: True


# Step 4: Multi-Modal Masking & Cross-Attention Control
To support interleaved text and images, text tokens should only cross-attend to the visual features of the image that immediately precedes them in the context. We construct a mask to enforce this constraint during the cross-attention step.

In [5]:
class InterleavedGatedCrossAttention(nn.Module):
    def __init__(self, dim: int, dim_visual: int, heads: int = 8, dim_head: int = 64):
        super().__init__()
        self.heads = heads
        self.scale = dim_head ** -0.5
        self.dim_head = dim_head
        inner_dim = dim_head * heads

        self.norm_text = nn.LayerNorm(dim)
        self.norm_visual = nn.LayerNorm(dim_visual)

        self.to_q = nn.Linear(dim, inner_dim, bias=False)
        self.to_kv = nn.Linear(dim_visual, inner_dim * 2, bias=False)
        self.to_out = nn.Linear(inner_dim, dim, bias=False)

        self.tanh_gate = nn.Parameter(torch.zeros(1))

    def forward(
        self,
        x: torch.Tensor,
        y: torch.Tensor,
        media_locations: torch.Tensor
    ) -> torch.Tensor:
        """
        x: text features [batch, text_len, dim]
        y: visual features from resampler [batch, num_media, queries_per_media, dim_visual]
        media_locations: boolean mask [batch, text_len] indicating which media index each text token should attend to
        """
        residual = x
        x = self.norm_text(x)

        # Reshape visual inputs to flatten media indices: [batch, num_media * queries_per_media, dim_visual]
        b, num_media, q_per_m, d_vis = y.shape
        y = y.view(b, num_media * q_per_m, d_vis)
        y = self.norm_visual(y)

        h = self.heads
        q = self.to_q(x).view(b, x.shape[1], h, self.dim_head).transpose(1, 2)
        k, v = self.to_kv(y).chunk(2, dim=-1)
        k = k.view(b, -1, h, self.dim_head).transpose(1, 2)
        v = v.view(b, -1, h, self.dim_head).transpose(1, 2)

        # Dot product attention
        dots = torch.matmul(q, k.transpose(-1, -2)) * self.scale

        # Create the mask:
        # Each text token 'i' attends only to media index indicated by media_locations[b, i]
        # media_locations has shape [batch, text_len], containing integer index of the media (0 to num_media-1), or -1 if no media precedes
        # We construct a mask of shape [batch, 1, text_len, num_media * queries_per_media]
        seq_len = x.shape[1]

        # Expand media locations to align with visual tokens
        # [batch, text_len, 1] compared to [batch, 1, num_media * queries_per_media]
        media_idx_per_token = media_locations.unsqueeze(-1) # [b, text_len, 1]

        # Create a matching tensor for visual media indices
        # e.g., if we have 3 images, indices are [0,0,..0, 1,1,..1, 2,2,..2]
        vis_media_indices = torch.arange(num_media, device=y.device).repeat_interleave(q_per_m)
        vis_media_indices = vis_media_indices.unsqueeze(0).unsqueeze(0) # [1, 1, num_media * q_per_m]

        # Construct attention mask: True where they match, and token is not pointing to -1
        mask = (media_idx_per_token == vis_media_indices) & (media_idx_per_token != -1)
        mask = mask.unsqueeze(1) # [batch, 1, text_len, num_media * queries_per_media] to broadcast across heads

        # Apply mask (fill unassociated regions with a large negative value)
        dots = dots.masked_fill(~mask, float('-inf'))

        # Softmax over key dimension
        attn = dots.softmax(dim=-1)

        # Handle case where a text token has no preceding media (-1), softmax of -inf returns NaNs
        attn = torch.nan_to_num(attn)

        out = torch.matmul(attn, v)
        out = out.transpose(1, 2).contiguous().view(b, seq_len, -1)
        out = self.to_out(out)

        return residual + out * self.tanh_gate.tanh()

# Quick validation
test_interleaved = InterleavedGatedCrossAttention(dim=256, dim_visual=256, heads=4)
text_features = torch.randn(1, 10, 256)
visual_features = torch.randn(1, 2, 8, 256) # 1 batch, 2 images, 8 tokens per image

# Token 0-4 attend to image 0; tokens 5-9 attend to image 1
media_locs = torch.tensor([[0, 0, 0, 0, 0, 1, 1, 1, 1, 1]])

out = test_interleaved(text_features, visual_features, media_locs)
print("Input text shape:", text_features.shape)
print("Output text shape:", out.shape)

Input text shape: torch.Size([1, 10, 256])
Output text shape: torch.Size([1, 10, 256])


# Step 5: The Flamingo Layer Wrapper
We wrap an existing decoder block of our Language Model. We insert our gated cross-attention module before the language model's original self-attention block, allowing the text features to update using visual cues from the interleaved images.

In [6]:
class FlamingoLayer(nn.Module):
    def __init__(self, lm_layer: nn.Module, dim: int, dim_visual: int, heads: int = 8, dim_head: int = 64):
        super().__init__()
        # Frozen language model decoder layer
        self.lm_layer = lm_layer

        # Interleaved Gated Cross-Attention and Feed-Forward
        self.gated_cross_attn = InterleavedGatedCrossAttention(
            dim=dim,
            dim_visual=dim_visual,
            heads=heads,
            dim_head=dim_head
        )
        self.gated_ff = GatedFeedForward(dim=dim)

    def forward(
        self,
        x: torch.Tensor,
        y: torch.Tensor,
        media_locations: torch.Tensor,
        *args,
        **kwargs
    ) -> torch.Tensor:
        """
        x: text features [batch, text_len, dim]
        y: visual features [batch, num_media, queries_per_media, dim_visual]
        media_locations: [batch, text_len] mapping text tokens to corresponding media index
        """
        # 1. Update text representations using visual inputs via Interleaved Gated Cross-Attention
        x = self.gated_cross_attn(x, y, media_locations)

        # 2. Refine the update with the Gated Feed-Forward
        x = self.gated_ff(x)

        # 3. Pass the resulting representation to the standard frozen language model decoder block
        # We forward optional positional arguments and attention masks used by the underlying LM layer
        out = self.lm_layer(x, *args, **kwargs)

        # Some Hugging Face layers return a tuple (hidden_states, optional_attentions), we extract hidden_states if so
        if isinstance(out, tuple):
            x_out = out[0]
            return (x_out,) + out[1:]

        return out

# Quick mock-up test
mock_lm_layer = nn.Identity() # Acting as our frozen LLM layer
flamingo_block = FlamingoLayer(lm_layer=mock_lm_layer, dim=256, dim_visual=256)

test_text = torch.randn(1, 10, 256)
test_visual = torch.randn(1, 2, 8, 256)
test_locs = torch.tensor([[0, 0, 0, 0, 0, 1, 1, 1, 1, 1]])

block_output = flamingo_block(test_text, test_visual, test_locs)
print("Input block shape:", test_text.shape)
print("Output block shape:", block_output.shape)

Input block shape: torch.Size([1, 10, 256])
Output block shape: torch.Size([1, 10, 256])


# Step 6: The Full Flamingo Model Class
This high-level container class integrates the Vision Encoder, the Perceiver Resampler, and the modified Language Model decoder stacks to construct the full visual language model.

In [7]:
class Flamingo(nn.Module):
    def __init__(
        self,
        vision_encoder: nn.Module,
        perceiver_resampler: PerceiverResampler,
        flamingo_layers: nn.ModuleList,
        lm_embeddings: nn.Module,
        lm_head: nn.Module
    ):
        super().__init__()
        self.vision_encoder = vision_encoder
        self.perceiver_resampler = perceiver_resampler
        self.layers = flamingo_layers
        self.lm_embeddings = lm_embeddings
        self.lm_head = lm_head

    def forward(
        self,
        tokens: torch.Tensor,
        images: torch.Tensor,
        media_locations: torch.Tensor
    ) -> torch.Tensor:
        """
        tokens: [batch, text_len] text token IDs
        images: [batch, num_media, channels, height, width] input images
        media_locations: [batch, text_len] mapping text tokens to image indices
        """
        b, num_media, c, h, w = images.shape

        # 1. Flatten images to extract visual features through the Vision Encoder
        images_flat = images.view(b * num_media, c, h, w)
        # Extract raw patch features, output shape example: [batch * num_media, patches_seq, vision_dim]
        patch_features = self.vision_encoder(images_flat)

        # 2. Resample the visual features to fixed length tokens using Perceiver Resampler
        # Output shape: [batch * num_media, num_queries, dim]
        resampled_features = self.perceiver_resampler(patch_features)

        # Reshape to keep batch and media sequence separate for interleaved cross-attention
        # Output shape: [batch, num_media, num_queries, dim]
        resampled_features = resampled_features.view(b, num_media, -1, resampled_features.shape[-1])

        # 3. Embed the text tokens
        x = self.lm_embeddings(tokens)

        # 4. Forward representations through our interleaved multi-modal layers
        for layer in self.layers:
            x = layer(x, resampled_features, media_locations)

        # 5. Output projection to vocab logits
        logits = self.lm_head(x)
        return logits

# Mock setup to verify end-to-end integration
mock_vision = lambda x: torch.randn(x.shape[0], 196, 512) # Mock vision outputting 196 patches
mock_resampler = PerceiverResampler(dim=512, depth=1, num_queries=10)

# Setup 2 stacked language model layers wrapped with our Flamingo multi-modal interface
wrapped_layers = nn.ModuleList([
    FlamingoLayer(lm_layer=nn.Identity(), dim=512, dim_visual=512),
    FlamingoLayer(lm_layer=nn.Identity(), dim=512, dim_visual=512)
])

mock_embed = nn.Embedding(1000, 512)
mock_head = nn.Linear(512, 1000)

flamingo_model = Flamingo(
    vision_encoder=mock_vision,
    perceiver_resampler=mock_resampler,
    flamingo_layers=wrapped_layers,
    lm_embeddings=mock_embed,
    lm_head=mock_head
)

# Input data: 1 batch, 2 interleaved images (each 3, 224, 224)
test_images = torch.randn(1, 2, 3, 224, 224)
test_tokens = torch.tensor([[101, 45, 12, 88, 9, 302, 11, 4, 99, 102]])
test_locs = torch.tensor([[0, 0, 0, 0, 0, 1, 1, 1, 1, 1]]) # First 5 tokens attend image 0, rest attend image 1

logits = flamingo_model(test_tokens, test_images, test_locs)
print("Logits shape:", logits.shape) # Expected: [1, 10, 1000] (batch, seq_len, vocab_size)

Logits shape: torch.Size([1, 10, 1000])


# Step 7: Dummy Training & Fine-Tuning Loop
To train or fine-tune our Flamingo model, we optimize a standard auto-regressive next-token prediction language modeling loss. Only the newly added parameters (Perceiver Resampler, Gated Cross-Attention, and Gated Feed-Forward layers) are typically updated while the main Language Model and Vision Encoder remain frozen.

In [8]:
import torch.optim as optim

# 1. Freeze the core language layers (in this mock, represented by the linear layers/Identity)
# In a real setup, we would freeze the Vision Encoder and the standard Pretrained Language Model.
# We only train the Perceiver Resampler and the Gated Attention/FF modules inside our Flamingo Layers.

# Let's count trainable parameters to verify
def print_trainable_parameters(model):
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    all_params = sum(p.numel() for p in model.parameters())
    print(f"Trainable params: {trainable_params:,} | All params: {all_params:,} | Percentage: {100 * trainable_params / all_params:.2f}%")

print_trainable_parameters(flamingo_model)

Trainable params: 10,485,228 | All params: 10,485,228 | Percentage: 100.00%


In [9]:
# 2. Define a training step
# We will use CrossEntropyLoss to compute loss on next-token predictions
criterion = nn.CrossEntropyLoss(ignore_index=-100) # -100 is the standard Hugging Face ignore index
optimizer = optim.AdamW(flamingo_model.parameters(), lr=1e-4)

# Create mock training batch
# Batch size = 2
batch_tokens = torch.randint(10, 1000, (2, 12)) # 2 sentences of length 12
batch_images = torch.randn(2, 2, 3, 224, 224) # 2 batches, each containing 2 interleaved images
batch_locs = torch.tensor([
    [0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1], # Batch 1: first 6 tokens belong to image 0, last 6 to image 1
    [0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1]  # Batch 2: first 3 tokens belong to image 0, last 9 to image 1
])

# Targets are the tokens shifted by 1
targets = batch_tokens.clone()
targets[:, :-1] = batch_tokens[:, 1:]
targets[:, -1] = -100 # Last token has no target next token

# Put model in training mode
flamingo_model.train()

# Forward pass
logits = flamingo_model(batch_tokens, batch_images, batch_locs)

# Compute loss: flatten logits and targets (aligning them for cross entropy)
loss = criterion(logits.view(-1, 1000), targets.view(-1))

print("Calculated Loss:", loss.item())

# Backward pass
optimizer.zero_grad()
loss.backward()

# Check if gradients flow to our Gated Cross Attention gate parameter
print("Gradient of tanh_gate in Layer 0:", wrapped_layers[0].gated_cross_attn.tanh_gate.grad)

# Optimizer step
optimizer.step()
print("Optimization step completed successfully!")

Calculated Loss: 7.071280002593994
Gradient of tanh_gate in Layer 0: tensor([0.0129])
Optimization step completed successfully!
